In [1]:
SUBPROCESS_NAME = "matmul"

In [2]:
import subprocess

completed = subprocess.run(
    ["python", f"cpu/{SUBPROCESS_NAME}.py"],
    check=True,
    text=True,
    capture_output=True,
)

print(completed.stdout)


## CPU matrix multiplication analysis

In [ ]:
import time

import numpy as np


BATCH_SIZES = (1, 32, 512, 2048)
INPUT_SIZE = 64
OUTPUT_SIZE = 64
DTYPE = np.int32
REPEATS = 10


def analyze_matmul(batch_size, repeats=REPEATS):
    rng = np.random.default_rng(42)
    activations = rng.integers(-8, 8, size=(batch_size, INPUT_SIZE), dtype=DTYPE)
    weights = rng.integers(-8, 8, size=(INPUT_SIZE, OUTPUT_SIZE), dtype=DTYPE)

    times = []
    output = None
    for _ in range(repeats):
        start = time.perf_counter()
        output = activations @ weights
        times.append(time.perf_counter() - start)

    return {
        "batch": batch_size,
        "shape": output.shape,
        "best_time_s": min(times),
        "avg_time_s": sum(times) / len(times),
        "checksum": int(output.sum()),
    }


In [ ]:
analysis = [analyze_matmul(batch_size) for batch_size in BATCH_SIZES]

for row in analysis:
    print(
        f"batch={row['batch']:<4} shape={row['shape']} "
        f"best={row['best_time_s']:.6f}s avg={row['avg_time_s']:.6f}s "
        f"checksum={row['checksum']}"
    )
